## Output Parsers ##

 ## Output Parsers->Convert raw LLM output into a format like json,pydantic,csv etc.Output parsers work with both type of models(models which can or can,t generate the structured output) ##

**1)StringOutputParser->Convert LLM response into the string.**

In [4]:
#Without StrOutputParser

import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=100
)

model = ChatHuggingFace(llm=llm)

template1 = ChatPromptTemplate.from_template("Write down detailed explanation of: {topic}")
prompt1 = template1.invoke({"topic": "Cricket"})
result1 = model.invoke(prompt1)

template2 = ChatPromptTemplate.from_template("Write down 5 line summary of:\n{topic}")
prompt2 = template2.invoke({"topic": result1.content})
result2 = model.invoke(prompt2)

print(result2.content)

Certainly! Here's a 5-line summary of cricket:

Cricket, originating in England, is a bat-and-ball game played between two teams of eleven players. The objective is to score runs by hitting the ball with a bat and running between wickets. The game is divided into innings, with players trying to score as many runs as possible while restricting the opposing team’s scoring.


## While accessing result.content works fine when calling model.invoke() directly in standard Python code, StrOutputParser() is required when building pipelines using LangChain Expression Language (LCEL).

Here are the key benefits to the point:

Enables LCEL Piping (|): You cannot pipe a object property like .content directly into a chain. StrOutputParser is a Runnable component, allowing you to build seamless chains like template1 | model | StrOutputParser() | template2 | model.

Native Streaming Support: When calling chain.stream(), StrOutputParser handles incoming AIMessageChunk objects dynamically and yields clean string tokens on the fly without breaking the generator.

Automatic Type Unwrapping: It strips away surrounding LLM metadata (token counts, response IDs, finish reasons, and tool call objects) and returns a plain Python str for downstream templates or functions.

Model Standardization: Provides a unified string extraction interface across all model providers (OpenAI, Gemini, Anthropic, Hugging Face), ensuring consistent behavior if you swap models.

In [7]:
#With StrOutputParser
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=100
)

model = ChatHuggingFace(llm=llm)
parser = StrOutputParser()

template1 = ChatPromptTemplate.from_template("Write down detailed explanation of: {topic}")

template2 = ChatPromptTemplate.from_template("Write down 5 line summary of: {text}")

chain = template1 | model | parser | template2 | model | parser

output = chain.invoke({"topic": "Cricket"})
print(output)



Cricket is a bat-and-ball game played between two teams of eleven players each. Widely regarded as the national sport of many Commonwealth countries, it is played from recreational to professional levels and even at the international stage. Believed to have originated in England between the 13th and 16th centuries, cricket has a rich and evolving history.


**JsonOutputParser**

**The flaw in the json output is that yo can not enforce any schema the LLM decide byself**

In [13]:
#JsonOutputParser


from langchain_core.output_parsers import JsonOutputParser
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=100
)

model = ChatHuggingFace(llm=llm)
parser = JsonOutputParser()

template = ChatPromptTemplate.from_template("Write down name,city and age of any fictional person \n {format_instruction}",partial_variables={"format_instruction":parser.get_format_instructions()})

chain = template | model | parser

result = chain.invoke({})
print(result)







{'name': 'Liam Carter', 'city': 'Newport', 'age': 28}


**StructuredOutputParser->you can get the desired output of json using responseschema**

**There is no data validation in Structureoupt**

In [2]:
import sys
!{sys.executable} -m pip install langchain



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=300
)

model = ChatHuggingFace(llm=llm)

schema = [
    ResponseSchema(name="fact_1", description="fact 1 about the topic"),
    ResponseSchema(name="fact_2", description="fact 2 about the topic"),
    ResponseSchema(name="fact_3", description="fact 3 about the topic"),
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = ChatPromptTemplate.from_template(
    "Write down three facts about the topic {topic}.\n{format_instructions}",
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({"topic": "Cricket"})

print(result)

ModuleNotFoundError: No module named 'langchain.output_parsers'